<a href="https://colab.research.google.com/github/SunSpot-Tech/Flyrank_Internship/blob/main/work/notebooks/%20%20w05_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/SunSpot-Tech/Flyrank_Internship/blob/main/work/notebooks/w05_model.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [ ]:
import duckdb, os, pandas as pd, numpy as np
from google.colab import userdata

os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
con = duckdb.connect()
con.sql("INSTALL httpfs;")
con.sql("LOAD httpfs;")
con.sql(f"""
CREATE OR REPLACE SECRET hf_token (TYPE HUGGINGFACE, TOKEN '{os.environ["HF_TOKEN"]}');
""")

FACT = "hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/*/*.parquet"
DIM  = "hf://datasets/FlyRank/internship-warehouse/dim_content.parquet"

feature_frame = con.sql(f"""
    SELECT
        f.content_hash_id, f.client_hash_id,
        DATE '2026-03-16' - d.content_updated_date AS days_since_last_update,
        SUM(CASE WHEN f.report_date < DATE '2026-03-16' THEN f.gsc_impressions ELSE 0 END) AS impressions_first_half,
        SUM(CASE WHEN f.report_date < DATE '2026-03-16' THEN f.gsc_clicks ELSE 0 END) AS clicks_first_half,
        AVG(CASE WHEN f.report_date < DATE '2026-03-16' THEN f.gsc_avg_position END) AS avg_position_first_half,
        SUM(CASE WHEN f.report_date >= DATE '2026-03-16' THEN f.gsc_clicks ELSE 0 END) AS clicks_second_half
    FROM read_parquet('{FACT}', hive_partitioning=1) f
    JOIN read_parquet('{DIM}') d ON f.content_hash_id = d.content_hash_id
    WHERE f.month = '2026-03' AND f.gsc_data_available IS TRUE
    GROUP BY f.content_hash_id, f.client_hash_id, d.content_updated_date
""").df()

# Same fix as Week 4: snapshot date issue
feature_frame['days_since_last_update'] = feature_frame['days_since_last_update'].where(
    feature_frame['days_since_last_update'] >= 0, np.nan
)
feature_frame['is_declining'] = (feature_frame.clicks_second_half < feature_frame.clicks_first_half).astype(int)
feature_frame['ctr_first_half'] = feature_frame.clicks_first_half / feature_frame.impressions_first_half.replace(0, np.nan)

pos_bins = [0, 3, 6, 10, 20, 1000]
pos_labels = ['1-3', '4-6', '7-10', '11-20', '20+']
feature_frame['position_tier'] = pd.cut(feature_frame.avg_position_first_half, bins=pos_bins, labels=pos_labels)
tier_avg = feature_frame.groupby('position_tier', observed=True)['ctr_first_half'].mean()
feature_frame['tier_avg_ctr'] = feature_frame['position_tier'].map(tier_avg).astype(float)
feature_frame['ctr_gap'] = feature_frame['ctr_first_half'] < feature_frame['tier_avg_ctr']

print(feature_frame.shape)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

(176738, 12)


## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*
## 1. Method Choice and Why

**Method: Decision Tree, then Random Forest for comparison.**

I'm choosing a tree-based method over logistic regression because Week 4's
signal audit found both my candidate signals (staleness, CTR gap) behave
**OPPOSITE** of their naive hypothesis, and the relationship differs by
position tier (the CTR-gap effect size varies a lot across tiers — from
5.5% to 70.8% decline rate depending on tier). A linear model would need
me to hand-engineer that interaction; a tree naturally splits on it.

I start with a single Decision Tree (max_depth=3-4) for readability — the
Week 2 lesson was that a shallow tree is still something I can read and
explain, unlike a black box. I then compare it to a Random Forest to see
whether the added complexity earns its keep, per the "does not reward
complexity alone" requirement — if the Random Forest doesn't clearly beat
the single tree, I keep the simpler model.

## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

**Grouped by client (`client_hash_id`), not random, and not time-based.**

Multiple pages belong to the same client, and clients likely share
systematic behavior (industry, content strategy, GSC integration quality).
A random row-level split would let pages from the same client appear in both train and test, silently leaking client-level patterns and inflating my score — exactly the risk flagged in the Week 1 lane guide. A client-grouped split (GroupShuffleSplit on `client_hash_id`) ensures every client's pages fall entirely into either train or test, never split across both.

I am not using a time-based split here because my feature/label windows
are already split within the same March month (first half → features,
second half → label) — the grouping risk that matters most for this
lane is client leakage, not time leakage, at this stage.

In [ ]:
from sklearn.model_selection import GroupShuffleSplit

features = ['impressions_first_half', 'clicks_first_half', 'avg_position_first_half',
            'ctr_first_half', 'days_since_last_update']

model_df = feature_frame.dropna(subset=['avg_position_first_half']).copy()
X = model_df[features].fillna(-1)  # -1 sentinel for missing days_since_last_update
y = model_df['is_declining']
groups = model_df['client_hash_id']

gss = GroupShuffleSplit(n_splits=1, test_size=0.3, random_state=42)
train_idx, test_idx = next(gss.split(X, y, groups=groups))

X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]
test_df = model_df.iloc[test_idx].copy()

print(f"Train: {len(X_train)} rows, {groups.iloc[train_idx].nunique()} clients")
print(f"Test:  {len(X_test)} rows, {groups.iloc[test_idx].nunique()} clients")
print(f"Overlap check (should be 0): {len(set(groups.iloc[train_idx]) & set(groups.iloc[test_idx]))}")

Train: 110403 rows, 30 clients
Test:  41578 rows, 14 clients
Overlap check (should be 0): 0


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

In [ ]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    return np.asarray(labels)[order[:k]].mean()

# --- Baseline rule, applied to TEST split only (same rows, same metric) ---
def assign_reason(row):
    is_stale = pd.notna(row['days_since_last_update']) and row['days_since_last_update'] >= 180
    is_visible = row['impressions_first_half'] >= 500
    has_ctr_gap = row['ctr_gap']
    if is_stale and is_visible and has_ctr_gap:
        return 2  # STALE_AND_CTR_GAP
    elif is_stale and is_visible:
        return 1
    elif has_ctr_gap:
        return 1
    else:
        return 0

test_df['baseline_flag_score'] = test_df.apply(assign_reason, axis=1) * test_df['impressions_first_half']

# --- Decision Tree ---
tree = DecisionTreeClassifier(max_depth=4, class_weight='balanced', random_state=42)
tree.fit(X_train, y_train)
tree_scores = tree.predict_proba(X_test)[:, 1]

# --- Random Forest ---
rf = RandomForestClassifier(n_estimators=200, max_depth=6, class_weight='balanced', random_state=42)
rf.fit(X_train, y_train)
rf_scores = rf.predict_proba(X_test)[:, 1]

results = []
for k in (20, 50, 100):
    results.append({
        'K': k,
        'Baseline rule': round(precision_at_k(test_df['baseline_flag_score'], y_test, k), 3),
        'Decision Tree': round(precision_at_k(tree_scores, y_test, k), 3),
        'Random Forest': round(precision_at_k(rf_scores, y_test, k), 3),
    })

comparison_table = pd.DataFrame(results)
print(comparison_table)

     K  Baseline rule  Decision Tree  Random Forest
0   20           0.30           0.95           1.00
1   50           0.32           0.90           0.92
2  100           0.42           0.86           0.88


| K   | Baseline rule | Decision Tree | Random Forest |
|-----|--------------|----------------|----------------|
| 20  | 0.30         | 0.95           | 1.00           |
| 50  | 0.32         | 0.90           | 0.92           |
| 100 | 0.42         | 0.86           | 0.88           |

**Both models clearly beat the Week 4 baseline rule at every K** — the
gap is large (e.g. at K=20: 0.30 baseline vs 0.95-1.00 for the models).
This makes sense given Week 4's finding: the hand-written rule was built
on signals (staleness, CTR gap) that turned out to point OPPOSITE of the
hypothesis, so a rule built on the wrong-direction assumption performs
only slightly better than chance (0.30-0.42 against a 54% base decline
rate is actually *worse* than a lazy always-decline guess in some
ranges). A model that learns the *actual* direction of these signals
from data — rather than assuming a direction — recovers real predictive
power.

**Random Forest vs Decision Tree:** Random Forest edges out the single
tree at every K, but the gap is small (e.g. 1.00 vs 0.95 at K=20, 0.92
vs 0.90 at K=50) and narrows further at K=100 (0.88 vs 0.86). Given the
"does not reward complexity alone" principle, this is a case where the
added complexity gives a modest, real improvement — not a required one.
I keep Random Forest as my primary model since the gain is consistent
across all three K values (not just a lucky K=20), but the single
Decision Tree remains a strong, more interpretable fallback if
explainability matters more than the last few points of precision.

**One caution:** Precision@20 = 1.00 for Random Forest is a strong
number worth treating carefully — I check in Section 4 whether this is
genuine signal or a sign the model is leaning too hard on one feature
that happens to work well in this particular test slice.

## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

In [ ]:
from sklearn.tree import export_text
print(export_text(tree, feature_names=features))

|--- ctr_first_half <= 0.00
|   |--- clicks_first_half <= 0.50
|   |   |--- class: 0
|   |--- clicks_first_half >  0.50
|   |   |--- class: 0
|--- ctr_first_half >  0.00
|   |--- ctr_first_half <= 0.00
|   |   |--- ctr_first_half <= 0.00
|   |   |   |--- ctr_first_half <= 0.00
|   |   |   |   |--- class: 1
|   |   |   |--- ctr_first_half >  0.00
|   |   |   |   |--- class: 1
|   |   |--- ctr_first_half >  0.00
|   |   |   |--- ctr_first_half <= 0.00
|   |   |   |   |--- class: 0
|   |   |   |--- ctr_first_half >  0.00
|   |   |   |   |--- class: 1
|   |--- ctr_first_half >  0.00
|   |   |--- impressions_first_half <= 75.50
|   |   |   |--- ctr_first_half <= 0.06
|   |   |   |   |--- class: 1
|   |   |   |--- ctr_first_half >  0.06
|   |   |   |   |--- class: 1
|   |   |--- impressions_first_half >  75.50
|   |   |   |--- impressions_first_half <= 874.50
|   |   |   |   |--- class: 1
|   |   |   |--- impressions_first_half >  874.50
|   |   |   |   |--- class: 1



In [ ]:
from sklearn.inspection import permutation_importance

perm = permutation_importance(rf, X_test, y_test, n_repeats=10, random_state=42, scoring='roc_auc')
importance_df = pd.DataFrame({
    'feature': features,
    'importance_mean': perm.importances_mean,
    'importance_std': perm.importances_std
}).sort_values('importance_mean', ascending=False)
print(importance_df)

                   feature  importance_mean  importance_std
3           ctr_first_half         0.172428        0.002393
1        clicks_first_half         0.015255        0.000850
0   impressions_first_half         0.004976        0.000508
2  avg_position_first_half         0.003155        0.000266
4   days_since_last_update         0.001303        0.000266


In [ ]:
test_df['rf_score'] = rf_scores
test_df['rf_pred'] = (rf_scores >= 0.5).astype(int)

false_positives = test_df[(test_df['rf_pred'] == 1) & (test_df['is_declining'] == 0)]
false_negatives = test_df[(test_df['rf_pred'] == 0) & (test_df['is_declining'] == 1)]

print(f"False positives: {len(false_positives)}")
print(false_positives[features].describe())
print(f"\nFalse negatives: {len(false_negatives)}")
print(false_negatives[features].describe())

False positives: 6884
       impressions_first_half  clicks_first_half  avg_position_first_half  \
count             6884.000000        6884.000000              6884.000000   
mean              2513.112580           9.863016                 5.857463   
std               4395.275876          27.724611                 5.347929   
min                  5.000000           1.000000                 0.350984   
25%                535.750000           1.000000                 3.082710   
50%               1191.000000           3.000000                 4.453653   
75%               2625.000000           8.000000                 6.530924   
max              78162.000000        1165.000000                60.318182   

       ctr_first_half  days_since_last_update  
count     6884.000000             1432.000000  
mean         0.005241               19.132682  
std          0.009102                5.216004  
min          0.000067                7.000000  
25%          0.001581               19.00000

## 4. Errors and Interpretation

**Feature reliance:** `ctr_first_half` dominates permutation importance
(0.172), roughly 10x the next-highest feature (`clicks_first_half`,
0.015). The decision tree confirms this — its very first split is
`ctr_first_half <= 0.00 → class: 0`, and most subsequent splits also
threshold on near-zero CTR values.

**This is a red flag, not a win.** `is_declining` is defined as
`clicks_second_half < clicks_first_half`. When `clicks_first_half = 0`,
CTR is mechanically exactly 0, and `is_declining` is *guaranteed* to be
0 — clicks cannot go negative. So low/zero CTR doesn't predict future
decline; it's a floor effect baked directly into how the label is
constructed. My Precision@20 = 1.00 for Random Forest is likely inflated
by the model exploiting this arithmetic boundary on low-volume pages,
not by learning genuine content-performance signal.

**False positives (6,884 rows):** average 9.9 clicks, 0.0052 CTR,
2,513 impressions in the first half — mid-volume pages the model
expected to decline but didn't. These sit above the zero-click floor,
so they're a more genuine test of the model's judgment, and it's wrong
here more often than the headline Precision@K suggests.

**False negatives (1 row):** a single page with 1 click, near-zero CTR
— an edge case where "decline" required dropping from 1 click to 0,
an unstable, low-volume signal to predict from.

**Honest conclusion:** the model-vs-baseline comparison in Section 3 is
real and directionally informative — both models substantially
outperform the hand-written baseline rule. But the specific Precision@20
= 1.00 number should not be reported as-is without this caveat: a
meaningful share of that precision likely comes from low-volume pages
where the label is mechanically determined by the feature, not
predicted. A next iteration should exclude or separately analyze pages
with `clicks_first_half` below some minimum threshold (e.g. ≥5), so the
metric reflects genuine predictive power rather than an artifact of the
label's own construction.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.